In [ ]:
# import all relevant modules
import os
from pathlib import Path

import pandas as pd
import requests

## Constant definition

The dataset from PFAS Hub is located on [the download page](https://pdh.cnrs.fr/en/download/). The direct download URL is set here for easy adjustability.
The same also applies to the target file in the `data/` folder.

In [ ]:
DATASET_URL = "https://pdh.cnrs.fr/download/full.parquet"
DATASET_FILE = Path("..") / "data" / "export.parquet"

### Download file

To save storage locally and to keep reproducibility, the notebook first checks if the export is already downloaded before the effective download starts.

In [ ]:
if not os.path.exists(DATASET_FILE):
    with requests.get(DATASET_URL, stream=True) as r:
        r.raise_for_status()
        with open(DATASET_FILE, 'wb') as f:
            f.writelines(r.iter_content(chunk_size=8192))

Load the dataset into pandas for further analysis.

In [ ]:
df_export = pd.read_parquet(DATASET_FILE)
df_export.shape

In [ ]:
df_export["sector"].value_counts()

In [ ]:
df_export[df_export["sector"].notna()].head(10)

In [ ]:
print(df_export.columns)

In [ ]:
import datetime as dt

from pfas_import import storage as s

df_export.to_parquet(path=f"s3://raw/{dt.datetime.now(dt.UTC).isoformat()}-import.parquet", filesystem=s.get_s3_fs())